# TabNet ONNX Inference Examples

This notebook demonstrates practical usage of TabNet ONNX models for inference in production environments.

**Date:** 2025-09-30

---

## Overview

Practical examples covering:
- Single sample prediction
- Batch processing
- Real data inference
- Performance benchmarking
- Results export


## Imports and Setup

In [ ]:
import json
import time
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import onnxruntime as rt

print("All imports successful")

## TabNet ONNXPredictor Class

In [ ]:
class TabNetONNXPredictor:
    """Predictor class for TabNet ONNX models."""

    def __init__(self, model_dir: Path = Path("onnx_models")):
        """
        Initialize predictor with ONNX models.

        Args:
            model_dir: Directory containing ONNX models
        """
        self.model_dir = Path(model_dir)

        # Load scaler
        scaler_path = self.model_dir / "scaler.onnx"
        self.scaler_sess = rt.InferenceSession(str(scaler_path))
        self.scaler_input_name = self.scaler_sess.get_inputs()[0].name
        self.scaler_output_name = self.scaler_sess.get_outputs()[0].name

        # Load model
        model_path = self.model_dir / "model.onnx"
        self.model_sess = rt.InferenceSession(str(model_path))
        self.model_input_name = self.model_sess.get_inputs()[0].name

        # Load metadata
        metadata_path = self.model_dir / "onnx_metadata.json"
        with open(metadata_path, 'r') as f:
            self.metadata = json.load(f)

        self.n_features = self.metadata['n_features']

        print(f"TabNet ONNX Predictor initialized")
        print(f"  Features: {self.n_features}")
        print(f"  Model type: {self.metadata['model_type']}")

    def predict(
        self,
        X: np.ndarray,
        return_probabilities: bool = True
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Make predictions on input data.

        Args:
            X: Input features, shape (n_samples, n_features)
            return_probabilities: Whether to return class probabilities

        Returns:
            Tuple of (predictions, probabilities)
        """
        if X.shape[1] != self.n_features:
            raise ValueError(
                f"Expected {self.n_features} features, got {X.shape[1]}"
            )

        X = X.astype(np.float32)

        # Scale input
        X_scaled = self.scaler_sess.run(
            [self.scaler_output_name],
            {self.scaler_input_name: X}
        )[0]

        # Predict
        probabilities = self.model_sess.run(None, {self.model_input_name: X_scaled})[0]
        predictions = (probabilities[:, 1] > 0.5).astype(int)

        if return_probabilities:
            return predictions, probabilities
        else:
            return predictions, None

    def predict_single(self, features: np.ndarray) -> dict:
        """
        Predict on a single sample.

        Args:
            features: Feature vector

        Returns:
            Dictionary with prediction results
        """
        if features.ndim == 1:
            features = features.reshape(1, -1)

        predictions, probabilities = self.predict(features)

        result = {
            'prediction': int(predictions[0]),
            'prediction_label': 'Land' if predictions[0] == 1 else 'Ocean',
            'confidence': float(np.max(probabilities[0])),
            'probabilities': {
                'Ocean': float(probabilities[0][0]),
                'Land': float(probabilities[0][1])
            }
        }

        return result

print("TabNetONNXPredictor class defined")

## Initialize Predictor

In [ ]:
predictor = TabNetONNXPredictor()

## Example 1: Single Sample Prediction

In [ ]:
print("="*70)
print("EXAMPLE 1: Single Sample Prediction")
print("="*70)

# Create random sample
sample = np.random.randn(predictor.n_features).astype(np.float32)

# Predict
result = predictor.predict_single(sample)

print(f"\nPrediction Results:")
print(f"  Class: {result['prediction']} ({result['prediction_label']})")
print(f"  Confidence: {result['confidence']:.4f}")
print(f"  Probabilities:")
print(f"    Ocean: {result['probabilities']['Ocean']:.4f}")
print(f"    Land:  {result['probabilities']['Land']:.4f}")

## Example 2: Batch Prediction

In [ ]:
print("\n" + "="*70)
print("EXAMPLE 2: Batch Prediction")
print("="*70)

# Create batch
batch_size = 100
X_batch = np.random.randn(batch_size, predictor.n_features).astype(np.float32)

print(f"\nProcessing batch of {batch_size} samples...")

# Predict
predictions, probabilities = predictor.predict(X_batch)

# Analyze
n_ocean = np.sum(predictions == 0)
n_land = np.sum(predictions == 1)
avg_confidence = np.mean(np.max(probabilities, axis=1))

print(f"\nBatch Results:")
print(f"  Total samples: {batch_size}")
print(f"  Ocean predictions: {n_ocean} ({n_ocean/batch_size*100:.1f}%)")
print(f"  Land predictions:  {n_land} ({n_land/batch_size*100:.1f}%)")
print(f"  Average confidence: {avg_confidence:.4f}")

# Show first 10
print(f"\n  First 10 predictions:")
for i in range(min(10, batch_size)):
    label = "Land" if predictions[i] == 1 else "Ocean"
    conf = probabilities[i][int(predictions[i])]
    print(f"    Sample {i+1}: {label:5s} (confidence: {conf:.4f})")

## Example 3: Real Test Data Prediction

In [ ]:
print("\n" + "="*70)
print("EXAMPLE 3: Prediction with Real Test Data")
print("="*70)

# Load real test data
test_data_path = Path("test_data.parquet")

if test_data_path.exists():
    print(f"\nLoading real test data...")
    X_test = pd.read_parquet(test_data_path)
    
    # Use first 10 samples
    X_sample = X_test.iloc[:10].values.astype(np.float32)
    
    print(f"Loaded {len(X_sample)} samples")
    print(f"\nRunning predictions...")
    
    # Predict
    predictions, probabilities = predictor.predict(X_sample)
    
    print(f"\nReal Data Predictions:")
    print(f"\n{'Sample':<10} {'Prediction':<15} {'Confidence':<15} {'Ocean Prob':<15} {'Land Prob':<15}")
    print("-" * 70)
    
    for i in range(len(X_sample)):
        label = "Land" if predictions[i] == 1 else "Ocean"
        conf = probabilities[i][int(predictions[i])]
        p_ocean = probabilities[i][0]
        p_land = probabilities[i][1]
        
        print(f"{i+1:<10} {label:<15} {conf:<15.4f} {p_ocean:<15.4f} {p_land:<15.4f}")
else:
    print(f"\nTest data not found at: {test_data_path}")
    print("Skipping real data example.")

## Example 4: Performance Benchmark

In [ ]:
print("\n" + "="*70)
print("EXAMPLE 4: Performance Benchmark")
print("="*70)

batch_sizes = [1, 10, 100, 1000]

print(f"\nBenchmarking inference speed...\n")
print(f"{'Batch Size':<15} {'Time (ms)':<15} {'Throughput':<20}")
print("-" * 50)

for batch_size in batch_sizes:
    X = np.random.randn(batch_size, predictor.n_features).astype(np.float32)

    # Warm-up
    predictor.predict(X)

    # Benchmark
    start = time.time()
    n_runs = 100 if batch_size <= 100 else 10
    for _ in range(n_runs):
        predictor.predict(X)
    elapsed = time.time() - start

    avg_time_ms = (elapsed / n_runs) * 1000
    throughput = batch_size * n_runs / elapsed

    print(f"{batch_size:<15} {avg_time_ms:<15.2f} {throughput:>10,.0f} samples/s")

## Example 5: Saving Predictions

In [ ]:
print("\n" + "="*70)
print("EXAMPLE 5: Saving Predictions to File")
print("="*70)

# Create sample data
n_samples = 100
X_data = np.random.randn(n_samples, predictor.n_features).astype(np.float32)

# Make predictions
predictions, probabilities = predictor.predict(X_data)

# Create results DataFrame
results_df = pd.DataFrame({
    'sample_id': range(1, n_samples + 1),
    'prediction': predictions,
    'prediction_label': ['Land' if p == 1 else 'Ocean' for p in predictions],
    'confidence': np.max(probabilities, axis=1),
    'prob_ocean': probabilities[:, 0],
    'prob_land': probabilities[:, 1]
})

# Save
output_path = Path("onnx_models") / "predictions.csv"
results_df.to_csv(output_path, index=False)

print(f"\nSaved {len(results_df)} predictions to: {output_path}")
print(f"\nSample of saved predictions:\n")
print(results_df.head(10).to_string(index=False))

## Examples Summary

In [ ]:
print("\n" + "="*70)
print("ALL EXAMPLES COMPLETED")
print("="*70)

print(f"\nExamples demonstrated:")
print(f"  Example 1: Single sample prediction")
print(f"  Example 2: Batch prediction")
print(f"  Example 3: Real test data prediction")
print(f"  Example 4: Performance benchmarking")
print(f"  Example 5: Saving predictions to file")

print(f"\nKey Takeaways:")
print(f"  - Use TabNetONNXPredictor class for easy inference")
print(f"  - Input must be float32 with {predictor.n_features} features")
print(f"  - Batch processing is efficient for multiple samples")
print(f"  - Results include predictions and probabilities")
print(f"\nReady for production deployment!")